# Interbank Contagion & Systemic Risk Analysis
This notebook builds a small synthetic banking system, models interbank lending, measures contagion with a simple DebtRank model, and compares failure scenarios using Monte Carlo simulation.

## Setup
Import the project functions and set the project paths.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

# The notebook is in notebooks/, so its parent is the project root.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from build_network import build_lending_network, save_edge_list
from create_plots import create_all_plots
from debtrank import calculate_debtrank, rank_banks_by_systemic_impact
from generate_bank_data import create_bank_data
from monte_carlo import run_monte_carlo, summarise_results

## Step 1: Synthetic bank data
We create five large core banks and fifteen smaller peripheral banks. Capital is each bank's initial loss-absorbing cushion.

In [ ]:
banks = create_bank_data()
banks.to_csv(PROJECT_ROOT / 'data' / 'banks.csv', index=False)
banks

## Step 2: Lending network
An arrow from A to B means A lends to B. Core banks lend to one another, while peripheral banks borrow from one or two core banks.

In [ ]:
network = build_lending_network(banks)
save_edge_list(network, PROJECT_ROOT / 'data' / 'lending_network.csv')
print(f'Banks: {network.number_of_nodes()}')
print(f'Lending relationships: {network.number_of_edges()}')

## Step 3: DebtRank contagion
We default one bank. Its lenders lose the relevant exposure relative to their capital, which can cause further distress. The final score is the share of system assets affected by distress.

In [ ]:
example_bank = 'Bank_01'
impact, distress = calculate_debtrank(network, example_bank)
print(f'Systemic impact of {example_bank}: {impact:.2%}')
pd.DataFrame({'bank_id': distress.keys(), 'distress': distress.values()}).sort_values('distress', ascending=False)

In [ ]:
ranking = rank_banks_by_systemic_impact(network)
ranking.to_csv(PROJECT_ROOT / 'outputs' / 'systemic_importance_ranking.csv', index=False)
ranking.head(10)

## Step 4: Monte Carlo simulation
We repeat the shock experiment 3,000 times. Random failure gives every bank the same chance of failure; size-weighted failure makes large banks more likely to fail.

In [ ]:
results = run_monte_carlo(network, number_of_simulations=3000)
results.to_csv(PROJECT_ROOT / 'outputs' / 'monte_carlo_results.csv', index=False)
summary = summarise_results(results)
summary.to_csv(PROJECT_ROOT / 'outputs' / 'monte_carlo_summary.csv', index=False)
summary

## Step 5: Visualisations
The figures show the lending network, the two loss distributions, and the ten banks with the greatest systemic impact.

In [ ]:
plot_paths = create_all_plots(network, results, ranking, PROJECT_ROOT / 'outputs')
for path in plot_paths.values():
    image = plt.imread(path)
    plt.figure(figsize=(10, 6))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

## Conclusion
Compare the random and size-weighted summary statistics above. In this synthetic system, failures that are more likely to involve large core banks should create larger systemic losses.